# Train YOLO26s Card Detector (640x640)

## What we're doing

Training a **YOLO26s** object detection model to recognize all 52 playing cards in real-time from a webcam. The model runs in the browser via ONNX Runtime Web inside [Magic Monitor](https://github.com/idvorkin/magic-monitor).

## Why YOLO?

YOLO (You Only Look Once) is a **fully convolutional** neural network — every layer is a convolution filter that slides across the image, so the same weights work at any input resolution. No fully-connected layers means no fixed input size. ([Good explainer](https://www.datacamp.com/blog/yolo-object-detection-explained))

## How transfer learning works here

1. **Start with pretrained weights** (`yolo26s.pt`) — trained on COCO (80 object classes: people, cars, dogs, etc.). The lower layers already know how to detect edges, textures, corners, and shapes.
2. **Fine-tune on playing cards** — the upper layers adapt from "this is a dog" to "this is a 7 of clubs". The lower-level feature detectors transfer directly.
3. **Export to ONNX** — bakes in the input resolution (640x640) for browser inference.

## Model size vs input resolution

| Setting | What it controls | Trade-off |
|---------|-----------------|-----------|
| **Model size** (n/s/m) | Network width & depth (number of parameters) | Bigger = more accurate, slower inference |
| **Input resolution** (416/640) | How many pixels the model sees | Higher = sees more detail, more compute |

Both are independent — a nano model at 640 has sharp eyes but a small brain. A medium model at 416 has a big brain but blurry vision. We use **small @ 640** as a good balance for browser inference.

## Dataset

~21,000 **synthetically generated** card images from [Roboflow](https://universe.roboflow.com/augmented-startups/playing-cards-ow27d) — card renders composited onto random backgrounds. 52 classes (standard deck, no jokers).

## Expected results

| Config | Params | Recall | Precision | Browser FPS |
|--------|--------|--------|-----------|-------------|
| YOLO26n @ 416 (old) | 2.5M | ~40% | ~95% | ~30 |
| YOLO26s @ 640 (new) | ~9M | ~60-75%* | ~95%* | ~10-15 |

*Estimated — run the benchmark cell at the end to get actual numbers.*

In [ ]:
# Install dependencies
!pip install -q ultralytics roboflow onnxruntime

In [ ]:
# Verify GPU is available
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
# Download dataset from Roboflow
from roboflow import Roboflow

# Get your free API key at https://app.roboflow.com/settings/api
ROBOFLOW_API_KEY = ""  # <-- paste your key here

if not ROBOFLOW_API_KEY:
    from google.colab import userdata
    try:
        ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
    except Exception:
        raise ValueError("Set ROBOFLOW_API_KEY above or in Colab secrets")

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("augmented-startups").project("playing-cards-ow27d")
version = project.version(4)
dataset = version.download("yolov8", location="./dataset")
print(f"\nDataset downloaded to ./dataset")

In [ ]:
# @title Training Configuration { run: "auto" }
MODEL_SIZE = "s"       # @param ["n", "s", "m"] {type: "string"}
IMGSZ = 640            # @param [416, 640] {type: "integer"}
EPOCHS = 50            # @param {type: "slider", min: 10, max: 200, step: 10}
BATCH = 16             # @param [8, 16, 32] {type: "integer"}

print(f"Training YOLO26{MODEL_SIZE} @ {IMGSZ}x{IMGSZ} for {EPOCHS} epochs, batch={BATCH}")

In [ ]:
# Train
from ultralytics import YOLO

model = YOLO(f"yolo26{MODEL_SIZE}.pt")

results = model.train(
    data="./dataset/data.yaml",
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project="./runs",
    name=f"cards-yolo26{MODEL_SIZE}",
    exist_ok=True,
)

print("\n✅ Training complete!")

In [ ]:
# Export to ONNX
import shutil
from pathlib import Path

best_weights = Path(f"./runs/cards-yolo26{MODEL_SIZE}/weights/best.pt")
model = YOLO(str(best_weights))

onnx_path = model.export(format="onnx", imgsz=IMGSZ, simplify=True)
output_path = "card-detector.onnx"
shutil.copy2(onnx_path, output_path)

size_mb = Path(output_path).stat().st_size / (1024 * 1024)
print(f"\n✅ ONNX exported: {output_path} ({size_mb:.1f} MB)")

In [ ]:
# Validate: check output shape and run on test image
import numpy as np
import onnxruntime as ort
import glob
import os

sess = ort.InferenceSession("card-detector.onnx")
inp = sess.get_inputs()[0]
out = sess.get_outputs()[0]
print(f"Input:  {inp.name} {inp.shape}")
print(f"Output: {out.name} {out.shape}")

# Dummy inference
dummy = np.random.randn(1, 3, IMGSZ, IMGSZ).astype(np.float32)
result = sess.run(None, {inp.name: dummy})[0]
print(f"Output shape: {result.shape}")
assert result.shape[2] == 6, f"Expected 6 features, got {result.shape[2]}"
print("✅ Output format correct: [1, N, 6] = [x1, y1, x2, y2, conf, class_id]")

In [ ]:
# Quick accuracy check on test set
import cv2

DATASET_LABELS = [
    "10C","10D","10H","10S","2C","2D","2H","2S","3C","3D","3H","3S",
    "4C","4D","4H","4S","5C","5D","5H","5S","6C","6D","6H","6S",
    "7C","7D","7H","7S","8C","8D","8H","8S","9C","9D","9H","9S",
    "AC","AD","AH","AS","JC","JD","JH","JS","KC","KD","KH","KS",
    "QC","QD","QH","QS",
]

test_images = sorted(glob.glob("dataset/test/images/*.jpg"))
input_name = sess.get_inputs()[0].name

total = correct = missed = fp = 0
for img_path in test_images:
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    # Letterbox resize
    scale = min(IMGSZ/w, IMGSZ/h)
    sw, sh = round(w*scale), round(h*scale)
    ox, oy = (IMGSZ-sw)//2, (IMGSZ-sh)//2
    resized = np.full((IMGSZ, IMGSZ, 3), 114, dtype=np.uint8)
    resized[oy:oy+sh, ox:ox+sw] = cv2.resize(img, (sw, sh))
    blob = resized.astype(np.float32) / 255.0
    blob = blob.transpose(2, 0, 1)[np.newaxis]
    output = sess.run(None, {input_name: blob})[0][0]
    dets = [(int(d[5]), float(d[4])) for d in output if d[4] >= 0.5]
    
    base = os.path.splitext(os.path.basename(img_path))[0]
    lp = os.path.join("dataset/test/labels", base + ".txt")
    gt = []
    if os.path.exists(lp):
        with open(lp) as f:
            gt = [int(l.split()[0]) for l in f if l.strip()]
    preds = sorted([d[0] for d in dets])
    for g in sorted(gt):
        total += 1
        if g in preds:
            correct += 1
            preds.remove(g)
        else:
            missed += 1
    fp += len(preds)

recall = correct/total*100 if total else 0
precision = correct/(correct+fp)*100 if (correct+fp) else 0
print(f"\n{'='*50}")
print(f"Test set: {len(test_images)} images, {total} ground truth cards")
print(f"Correct: {correct} | Missed: {missed} | False positives: {fp}")
print(f"Recall:    {recall:.1f}%")
print(f"Precision: {precision:.1f}%")
print(f"{'='*50}")

In [ ]:
# Upload to S3 (requires AWS CLI configured)
# Uncomment and run if you have AWS credentials set up:

# !aws s3 cp card-detector.onnx s3://idvorkin-models/card-detector.onnx
# print("✅ Uploaded to S3")

# Or download locally:
from google.colab import files
files.download("card-detector.onnx")